<a href="https://colab.research.google.com/github/fre-denni/genAIforUX-research/blob/main/logbook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install libraries

In [ ]:
!pip install -q pymupdf pdf2image google-genai spacy pandas scikit-learn pillow openpyxl
!python -m spacy download en_core_web_sm
!pip install timm einops

We are using an older version of transformers from hugging face to be sure that it's compatible with the Florence2 model from microsoft

### Import necessary libraries

In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
from google import genai
from google.genai import types
import spacy
from PIL import Image
import torch
import json
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from IPython.display import clear_output
clear_output()
print("Environment setup correctly!")

Setup folder



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#folder variables, change as you wish
main_folder = "logbook analysis"

# Define the base folder
base_folder = f"/content/drive/MyDrive/{main_folder}/"

# Define input and output folders
input_folder = os.path.join(base_folder, "logbook")
output_folder = os.path.join(base_folder, "output")
output_images_folder = os.path.join(base_folder, "output_images")

# Create output folders if they don't exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_images_folder, exist_ok=True)

print(f"Base folder: {base_folder}")
print(f"Input folder set to: {input_folder}")
print(f"Output folder created/verified: {output_folder}")
print(f"Output images folder created/verified: {output_images_folder}")


Add your API

In [ ]:

from google.colab import userdata
from google import genai

NOME_SECRET = 'GEMINI_API'

try:
    # Recupera la chiave
    api_key = userdata.get(NOME_SECRET)

    if not api_key:
        print(f"ERRORE: La chiave non è stata trovata. Controlla che il nome sia esattamente '{NOME_SECRET}' e che la levetta 'Notebook access' sia attiva.")
    else:
        # Stampiamo inizio e fine per scovare eventuali spazi vuoti o virgolette
        print(f"🔍 Controllo formato: La chiave inizia con '{api_key[:5]}...' e finisce con '...{api_key[-5:]}'")

        # Inizializza il client
        client = genai.Client(api_key=api_key)
        print("✅ Client inizializzato in locale.")

        # Facciamo una vera chiamata di test per validare la chiave sul server
        print("⏳ Provo a contattare Google AI Studio...")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents='Rispondi solo con "API funzionante al 100%!"'
        )
        print(f"🤖 Risposta da Gemini: {response.text}")

except userdata.SecretNotFoundError:
    print(f"❌ ERRORE CRITICO: Il secret chiamato '{NOME_SECRET}' non esiste nei tuoi Segreti di Colab.")
except Exception as e:
    print(f"❌ ERRORE API (Chiave invalida o problemi di rete): {e}")

Configure spacy

In [ ]:
nlp = spacy.load("en_core_web_sm") # English language
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add custom rules for design methodologies
patterns = [
    # A1 - User Journey Maps & varianti
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": "journey"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "ujm"}]},

    # A1 -Storyboards
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["storyboard", "storyboards"]}}]},

    # A2 - Mental Model Diagrams (cattura sia "mental model" che "mental model diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mental"}, {"LOWER": "model"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mmd"}]},

    # A2 -Mind & Empathy Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mind"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "empathy"}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A1 - User Jobs & JTBD
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["job", "jobs"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jobs"}, {"LOWER": "to"}, {"LOWER": "be"}, {"LOWER": "done"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jtbd"}]},

    # A1 - User Stories
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["story", "stories"]}}]},

    # A1 - Task Analysis (cattura sia "task analysis" che "task analysis diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "task"}, {"LOWER": "analysis"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},

    # A2 - Concept / Conceptual Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["concept", "conceptual"]}}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A3 - Research
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "literature"}, {"LOWER": "review"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "benchmarking"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["persona", "personas"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "digital"}, {"LOWER": "ethnography"}]},

    # A4 - Mapping (System, Service, Stakeholder) - cattura map, maps e mapping
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "system"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "service"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "stakeholder"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},

    # A2 - Affinity Diagram
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "affinity"}, {"LOWER": {"IN": ["diagram", "diagrams"]}}]}
]
ruler.add_patterns(patterns)

ruler.add_patterns(patterns)
print("Added custom config to spacy")

### Global function and prompts

Set up and force the JSON analysis for Gemini, and the various prompts for the tasks.

In [ ]:
# Force the config of output as json
json_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0 # more deterministic outputs
)

---

# And now the OCR loop!



In [ ]:
# Prompts
supervisor_prompt = """You are an expert UX Design Professor evaluating a student's logbook.
Your task is to look at the overall layout, headings, diagrams, and main text of this page to determine its MACRO-CONTEXT, and what is the page communicating.

Use the following STRICT SYLLABUS to classify the page:

- 'A1': Chrono Maps. Maps and UX analyses using TIME or sequential tasks as a core property. (Keywords: User Journey Map, UJM, Storyboard, User Jobs, JTBD, User Stories, Task Analysis).
- 'A2': Non-Chrono Maps. Qualitative maps of human complexity not based on time. (Keywords: Mental Model Diagram, MMD, Mind Map, Empathy Map, Concept Map, Conceptual Map, Affinity Diagram).
- 'A3': Design Research. Literature review, digital ethnography, and benchmarking to find systems/solutions for behavior change, controversial topics, or new tech adoption. (Keywords: Literature Review, Benchmarking, Personas, Digital Ethnography).
- 'A4': System Representation. Representing existing systems or new concepts derived from A3. (Keywords: System Map, Service Map, Service Blueprint, Stakeholder Map, App Mockup).
- 'AI': AI Usage & Reflections. A total review of AI tools used during the course, research with AI-powered tools, and personal reflections on the state of UX research regarding AI.
- 'Course': Course Reflections. General personal reflections on the UX design discipline or the course itself.
- 'Intro': Structural pages (Cover, Index, Acknowledgments, Thank You page).
- 'Unknown': If completely unclassifiable.

Determine the visual "Authorship" of the images on the page:
- 'Student': The diagrams, maps, or mockups appear to be actively created by the student (e.g., custom vector maps, Miro boards, structured templates).
- 'Reference': The visuals are clearly external references (e.g., screenshots of existing apps like Vinted/Patagonia, academic papers, website benchmarking).
- 'Mix': Both student creations and external references are present.
- 'No_image': No significant images present.

Return ONLY a valid JSON with this exact structure:
{
  "assignment": "A1",
  "key_content": "User Journey Map of Dott App",
  "content_is": "Student",
  "summary_content": "The student is mapping the temporal sequence of a user renting a scooter using the Dott App to find friction points.",
  "reasoning": "The heading says A1 and the visual is a sequential timeline map, fitting the Chrono Map definition."
}
"""

In [ ]:
# Analyse everything directly with VLM
def process_page(image_path, prompt):
  """Pass the image from the entire page to Gemini"""
  img = Image.open(image_path)

  try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[prompt,img],
        config=json_config
    )

    raw_text = response.text.strip()
    if raw_text.startswith("```json"):
      raw_text = raw_text[7:]
    if raw_text.endswith("```"):
      raw_text = raw_text[:-3]


    return json.loads(raw_text.strip())
  except Exception as e:
    print(f"Gemini API Error: {e}")
    return None

In [ ]:
def get_page_context(image_path):
  result = process_page(image_path, supervisor_prompt)
  if not result:
    return{
    "assignment": "idk",
    "key_content": "",
    "content_is": "",
    "summary_content": "",
    "reasoning": "API Error"
    }
  return result

In [ ]:
def extract_pipeline(image_path, context):
  dynamic_extractor_prompt = f"""You are an expert UX design document parser and Smart OCR.
    You are extracting data from a page that has ALREADY been classified with this context:

    - Assignment Bucket: {context.get('assignment', 'Unknown')}
    - Key Topic: {context.get('key_content', 'Unknown')}
    - Image Authorship: {context.get('content_is', 'Unknown')}
    - Context Summary: {context.get('summary_content', 'Unknown')}

    Use this context to accurately classify what you see. Perform 2 tasks sequentially:

    1. READ NARRATIVE TEXT: Extract the student's main narrative text, headings, and paragraphs.
       CRITICAL RULE: DO NOT extract text that is visually embedded inside screenshots, UI mockups, data charts, tables, or diagrams here.
       JSON SAFETY RULE: Strictly escape all double quotes (\\\") and newlines (\\n).
       If there is no readable narrative text, return an empty string "".

    2. LOCATE IMAGES AND EXTRACT EMBEDDED TEXT: Find significant diagrams, maps, tables, or mockups.
       MACRO-BOX RULE: If a schema spans a large area and contains sub-elements, draw ONE SINGLE bounding box around the ENTIRE macro-schema. Do NOT fragment it.

       For each macro-image found, provide:
       - 'type': Strictly classify the image (e.g., User Journey Map, Mental Model Diagram, UI Mockup, Data Table). Let the 'Key Topic' provided above guide you!
       - 'box_1000': The bounding box coordinates [ymin, xmin, ymax, xmax] using proportions from 0 to 1000.
       - 'embedded_text': ALL readable text strictly contained INSIDE that specific diagram/table. JSON SAFETY RULE: escape all quotes and newlines!

    Return ONLY a valid JSON with this exact structure:
    {{
      "text": "the extracted narrative text from the page...",
      "img": [
        {{
          "type": "User Journey Map",
          "box_1000": [100, 50, 450, 950],
          "embedded_text": "text read from inside this map..."
        }}
      ]
    }}
    """

  result = process_page(image_path, dynamic_extractor_prompt)
  if not result:
    print("Pipeline error")
    return{"text":"", "img":[]}
  return result

In [ ]:

def extract_pdf_pipeline(pdf_path, output_dir="output_images"):
  student_id = os.path.basename(pdf_path).replace(".pdf","")
  doc = fitz.open(pdf_path)
  os.makedirs(output_dir, exist_ok=True)
  master_records = []

  print(f"Processing {student_id} ({len(doc)} pages)...")

  for page_num in range(len(doc)):
      print(f" -> VLM analysis of page {page_num + 1}...")

      # High quality rendering of the page
      page = doc.load_page(page_num)
      pix = page.get_pixmap(dpi=250)
      img_page = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

      #Save the page
      temp_page_path = f"/tmp/temp_page_{student_id}_{page_num}.png"
      img_page.save(temp_page_path)

      #Recognize context
      print ("Getting context...")
      page_context = get_page_context(temp_page_path)

      assignment_label = page_context.get("assignment", "idk")
      content = page_context.get("key_content", "")
      content_is = page_context.get("content_is", "")
      reasoning = page_context.get("reasoning", "")

      print(f"     [Context Identified: {assignment_label} | {content} | {content_is}]")

      #Extract data
      print("Extracting data...")
      page_data = extract_pipeline(temp_page_path, page_context)

      extracted_text = page_data.get("text", "").strip()
      images = page_data.get("img", [])

      master_records.append({
          "ID": student_id,
          "Page": page_num + 1,
          "Layout": "Paragraph",
          "Scheme_type": "",
          "Text": extracted_text,
          "Path_Image": temp_page_path,
          "Assignment": assignment_label,
          "Context_Topic": content,
          "Context_Authorship": content_is,
          "Reasoning": reasoning,
      })

      # --- CUT IMAGES ---
      img_w, img_h = img_page.size
      PADDING = 50

      if images:
        print(f"Cropping {len(images)} context-aware schemas...")

      for idx, imm in enumerate(images):
        box = imm.get("box_1000")
        if box and len(box) == 4:
          ymin, xmin, ymax, xmax = box
          #Calculate coordinates
          base_left = (xmin / 1000.0) * img_w
          base_top = (ymin / 1000.0) * img_h
          base_right = (xmax / 1000.0) * img_w
          base_bottom = (ymax / 1000.0) * img_h
          #Add padding
          left = max(0, base_left - PADDING)
          top = max(0, base_top - PADDING)
          right = min(img_w, base_right + PADDING)
          bottom = min(img_h, base_bottom + PADDING)

          try:
            crop_img = img_page.crop((left, top, right, bottom))
            #clean file name
            entry_type = imm.get("type", "schema")
            type_clean = entry_type.replace(" ", "_").replace("-", "_").lower()
            type_clean = re.sub(r'[^a-z0-9_]', '',type_clean)
            type_clean = type_clean[:30]
            #if the string turns out empty
            if not type_clean:
              type_clean = "schema"
            #create filename
            filename = f"{student_id}_pag{page_num+1}_{type_clean}_{idx}.png"
            filepath = os.path.join(output_dir, filename)
            crop_img.save(filepath)

            embedded_text = imm.get("embedded_text", "").strip()

            #Save record
            master_records.append({
                "ID": student_id,
                "Page": page_num + 1,
                "Layout": "Image_Scheme",
                "Scheme_type": f"{entry_type}",
                "OCR_Text":f"{embedded_text}",
                "Path_Image": filepath,
                "Assignment": assignment_label,
                "Context_Topic": content,
                "Context_Authorship": content_is,
                "Reasoning": ""
            })
          except Exception as e:
            print(f"Errore ritaglio immagine a pag {page_num+1}: {e}")

  print(f"Complete PDF {student_id}")

  #create master dataframe
  df_master = pd.DataFrame(master_records)
  return df_master

Quick prova

In [ ]:
import os

proof = os.path.join(input_folder, "11167713.pdf") #quick and dirty 3mb logbook

if os.path.exists(proof):
  # Chiamiamo la funzione corretta per l'elaborazione dei PDF
  indexed = extract_pdf_pipeline(proof, output_dir=output_images_folder)

  # Ensure the 'results' directory exists before saving the CSV
  os.makedirs("results", exist_ok=True)

  #Save the result for security
  indexed.to_csv("results/master_index.csv", index=False)

  #Show preview
  display(indexed.head(100))
else:
  print(f"File not founded: {proof}")

Pulizia!

In [ ]:
nlp = spacy.load("en_core_web_sm")

def anonymizer(student_id, df_students):
  replacements ={
      "Margherita Pillan": "[PROFESSOR]",
      "Margherita": "[PROFESSOR]",
      "Pillan": "[PROFESSOR]",
      "Federico Denni": "[ASSISTANT]",
      "Federico": "[ASSISTANT]",
      "Denni": "[ASSISTANT]"
  }

  student_id = str(student_id)
  student_row = df_students[df_students['Codice persona'].astype(str) == student_id]

  if not student_row.empty:
    full_name = student_row.iloc[0]['Cognome-Nome'].title()
    parts = full_name.split()
    if len(parts) >= 2:
      cognome = parts[0]
      nome = " ".join(parts[1:])

      #all possible combinations
      replacements[f"{nome} {cognome}"] = student_id
      replacements[f"{cognome} {nome}"] = student_id
      replacements[nome] = student_id
      replacements[cognome] = student_id


  # Order the replacemetns
  sorted_replacements = dict(sorted(replacements.items(), key=lambda item: len(item[0]), reverse=True))


  def anonymize_text(text):
    if not isinstance(text, str) or not text.strip():
      return ""

    # Aimed subsitution
    for name, replacement in sorted_replacements.items():
      # Usa regex con word boundaries per non tagliare pezzi di altre parole
      pattern = r'\b' + re.escape(name) + r'\b'
      text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # Generic subsition
    doc = nlp(text)
    cleaned_text = text
    persons = sorted([ent.text for ent in doc.ents if ent.label_ == "PERSON"], key=len, reverse=True)

    for person in persons:
        if person != student_id and "[PROFESSOR]" not in person and "[ASSISTANT]" not in person:
           cleaned_text = cleaned_text.replace(person, "[STUDENT/PERSON]")

    return cleaned_text

  return anonymize_text


In [ ]:
# filtering prompt
text_clean_prompt = """You are an expert UX teaching assistant validating extracted text.

CONTEXT:
Assignment: {assignment}
Topic: {topic}
Supervisor Reasoning: {reasoning}
Detected Methods: {design_methodologies}
Detected Entities: {entities}

TEXT:
---
{text}
---

EVALUATION CRITERIA:
Assess if the text explicitly discusses:
1. A UX Design Methodology (Application or Explanation).
2. A Digital Tool (Figma, Miro, Python, etc.).
3. Reflections (Course, Tools, Methodology, AI).
4. AI Usage.

DECISION RULE:
If the text discusses AT LEAST ONE of the above criteria OR contains highly relevant design methods/entities detected by spaCy, set "keep" to true.
If the text DOES NOT discuss these criteria AND lacks relevant entities/methodologies (e.g., it's just an index, generic filler, or unreadable OCR), set "keep" to false.

Return ONLY a JSON:
{{
  "keep": true/false,
  "category": "Method Application" | "Method Explanation" | "Tool Usage" | "Reflection" | "Irrelevant",
  "reasoning": "Brief reason..."
}}
"""

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas(desc="Elaborazione LLM")

students_path = os.path.join(base_folder,"students-team-ux-grade.csv")
df_students = pd.read_csv(students_path)

df_logbook = pd.read_csv("results/master_index.csv")

print("Cleaning phase 1: Anonimity")
def apply_anonymizer(row):
  student_id = str(row['ID'])
  anon_func = anonymizer(student_id, df_students)

  #apply anonimization on text
  text_content = str(row['Text'])
  text = anon_func(text_content) if pd.notnull(row['Text']) and text_content.strip() != "" else ""
  return pd.Series(text)

df_logbook['Text'] = df_logbook.apply(apply_anonymizer, axis=1)

print("Cleaning phase 2: Images")
def check_image_exists(row):
  #skip no images
  if row['Layout'] != 'Image_Scheme':
    return True

  path = str(row['Path_Image'])
  #return true only if image present
  return os.path.exists(path)

df_logbook = df_logbook[df_logbook.apply(check_image_exists, axis=1)]

print("Cleaning phase 3: Entity extraction")
def extract_entities_and_methods(text):
  if not isinstance(text, str) or not text.strip():
    return "", ""

  doc = nlp(text)
  methods = set([ent.text for ent in doc.ents if ent.label_ == "DESIGN METHOD"])
  others = set([ent.text for ent in doc.ents if ent.label_ != "DESIGN METHOD"])

  return ", ".join(methods), ", ".join(others)

#Temporary column
df_logbook['full'] = df_logbook['Text'].fillna('') + " \n " + df_logbook['OCR_Text'].fillna('')

#Estraiamo e inseriamo nelle due nuove colonne
df_logbook[['entities', 'design_methodologies']] = df_logbook['full'].apply(
    lambda x : pd.Series(extract_entities_and_methods(x))
)

print("Cleaning phase 4: Validation and Classification of text")
def evaluate_text_with_llm(row):
  text_to_eval = row['full'].strip()

  # Discard text that is without entities
  if not text_to_eval and not row['entities'] and not row['design_methodologies']:
     return False

  prompt = text_clean_prompt.format(
      assignment=row.get('Assignment', 'Unknown'),
      topic=row.get('Context_Topic', 'Unknown'),
      reasoning=row.get('Reasoning', 'None'),
      design_methodologies=row.get('design_methodologies', 'None'),
      entities=row.get('entities', 'None'),
      text=text_to_eval
  )

  try:
    response = client.models.generate_content(
          model='gemini-2.5-flash',
          contents=prompt,
          config=json_config
    )

    # Pulizia della risposta markdown JSON
    raw_text = response.text.strip()
    if raw_text.startswith("```json"):
        raw_text = raw_text[7:]
    if raw_text.endswith("```"):
        raw_text = raw_text[:-3]

    result = json.loads(raw_text.strip())

    # Facciamo ritornare solo il booleano per decidere se tenere la riga
    return result.get("keep", False)

  except Exception as e:
    print(f"Gemini API Error for row page {row['Page']}: {e}")
    return True

# Usiamo progress_apply per vedere lo stato d'avanzamento, visto che chiamare le API richiederà qualche minuto
df_logbook['keep_flag'] = df_logbook.progress_apply(evaluate_text_with_llm, axis=1)

# Eliminiamo tutte le righe flaggate per l'eliminazione
df_cleaned = df_logbook[df_logbook['keep_flag'] == True].copy()

# Rimuoviamo le colonne temporanee usate per il processamento
df_cleaned = df_cleaned.drop(columns=['full', 'keep_flag'])

# Salviamo il risultato finale
os.makedirs("results", exist_ok=True)
df_cleaned.to_csv("results/master_index_cleaned.csv", index=False)

print("Operation completed!")
print(f"Original rows_ {len(df_logbook)}")
print(f"Remained rows: {len(df_cleaned)}")

display(df_cleaned.head())